# 项目 —— 航空 AI 助手

## 练习目标（理念）

把本周学过的 **Chat Completions、Gradio、Tool Calling、甚至 SQLite** 拼起来，做一个航空公司客服小助手：

- 用 **system message** 约束简短、礼貌、准确
- 用 **工具** 查询（和设置）机票价格
- 用 **Gradio ChatInterface** 提供可对话 UI

## 和本课 Day 4 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Tool / Function Calling | `tools=` + `finish_reason=="tool_calls"` |
| 多工具 / 循环调用 | `handle_tool_calls` + `while` |
| 结构化 tool schema | JSON Schema 描述参数 |
| 业务数据落地 | `sqlite3` 读写 `prices.db` |

## 怎么跑

1. 启动本地 Ollama，并确保已有 `gpt-oss:20b`（或改 `MODEL`）
2. 从上到下运行；先跑通「无工具聊天」，再跑「工具版」
3. 在 Gradio 里试：「How much is a ticket to London?」


In [ ]:
# ========== 导入：环境、JSON、OpenAI 兼容客户端、Gradio ==========

# os：环境变量
import os
# json：解析 tool_call.function.arguments
import json
# load_dotenv：读取 .env
from dotenv import load_dotenv
# OpenAI：这里指向本地 Ollama
from openai import OpenAI
# gradio：聊天 UI
import gradio as gr


In [ ]:
# ========== 初始化：加载环境；选用本地 Ollama 模型 ==========

# 把 .env 读入环境（override=True：文件优先）
load_dotenv(override=True)

# openai_api_key = os.getenv('OPENAI_API_KEY')
# 如果打开ai_api_key：
# print(f"OpenAI API 密钥存在并开始 {openai_api_key[:8]}")
# 别的：
# print("OpenAI API 密钥未设置")
    
# 型号 =“gpt-4.1-mini”
# openai = OpenAI()

# 作为替代方案，如果您想使用 Ollama 而不是 OpenAI
# 检查 Ollama 是否在本地为您运行（请参阅 week1/day2 练习），然后取消注释接下来的 2 行
# 本地模型名：需事先 ollama pull
MODEL = "gpt-oss:20b"
# 指向 Ollama 的 OpenAI 兼容 API；api_key 多为占位字符串
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [ ]:
# ========== system message：航空助手人设与回答约束（保留英文 prompt） ==========

system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""


In [ ]:
# ========== 无工具版聊天：只拼 messages，直接返回模型回复 ==========

def chat(message, history):
    # Gradio history → 标准 role/content 列表
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # system + 历史对话 + 当前用户句
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 调用本地模型（此时还没传 tools）
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    # 取出助手文本
    return response.choices[0].message.content

# 启动 Gradio；fn 指向上面的 chat
gr.ChatInterface(fn=chat).launch()


## 工具（Tools）

工具是前沿 LLM 很强的能力：你写一个 **Python 函数**，再给模型一份 **JSON Schema 说明书**，模型就可以在回复里提出「请帮我调用这个函数」。

听起来像把执行权交给模型？其实是：

1. 模型只**提议**调用（返回 `tool_calls`）
2. **你的代码**真正执行函数
3. 把结果以 `role: tool` 塞回 `messages`，再让模型生成最终自然语言回答


In [ ]:
# 打印 Gradio 版本，确认环境（不同大版本 ChatInterface API 细节可能不同）
print(gr.__version__)


In [ ]:
# ========== 示例业务函数：用字典模拟票价表 ==========

# 城市（小写）→ 价格字符串
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    # 调试：确认工具真的被调用了
    print(f"Tool called for city {destination_city}")
    # 查表；找不到就返回 Unknown...
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    # 返回给模型看的自然语言结果（不是直接给终端用户的最终话术）
    return f"The price of a ticket to {destination_city} is {price}"


In [ ]:
# 手动试跑工具函数：确认字典查询正常
get_ticket_price("London")


In [ ]:
# ========== 函数 schema：用 JSON Schema 描述给模型看 ==========

# 描述我们的函数需要一个特定的字典结构：
price_function = {
    # 必须与 Python 函数名 / 路由分支一致
    "name": "get_ticket_price",
    # 说明：模型靠它决定「该不该调用」
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}


In [ ]:
# ========== 包装成 tools 列表，供 create(..., tools=tools) 使用 ==========

# 这包含在工具列表中：
tools = [{"type": "function", "function": price_function}]


In [ ]:
# 查看 tools 结构，确认 schema 无误
tools


## 让模型「使用」我们的工具

流程要点：

1. 调用时传入 `tools=tools`
2. 若 `finish_reason == "tool_calls"`，说明模型想调用工具
3. 你执行工具 → 把结果消息追加进 `messages`
4. **再请求一次**模型，让它基于工具结果组织最终回答

下面先实现「单次工具调用」版本的 `chat`。


In [ ]:
# ========== 单工具调用版 chat：最多处理一轮 tool_calls ==========

def chat(message, history):
    # Gradio history → 标准消息
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # 拼完整对话
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 关键：把 tools 传给模型，它才知道有票价查询可用
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 若模型决定调用工具
    if response.choices[0].finish_reason=="tool_calls":
        # 取出带 tool_calls 的助手消息
        message = response.choices[0].message
        # 本地执行工具，得到一条 role=tool 的结果
        response = handle_tool_call(message)
        # 先追加助手的 tool_calls 消息
        messages.append(message)
        # 再追加工具结果
        messages.append(response)
        # 第二次请求：通常不再需要 tools（原代码如此，保留）
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content


In [ ]:
# ========== handle_tool_call：执行第一个 tool_call 并返回 tool 消息 ==========

# 我们必须编写该函数handle_tool_call：

def handle_tool_call(message):
    # 本版只取第一个工具调用
    tool_call = message.tool_calls[0]
    # 按名字路由到真正的 Python 函数
    if tool_call.function.name == "get_ticket_price":
        # arguments 是 JSON 字符串 → dict
        arguments = json.loads(tool_call.function.arguments)
        # 取出城市参数
        city = arguments.get('destination_city')
        # 查价
        price_details = get_ticket_price(city)
        # 组装 Chat Completions 要求的 tool 角色消息（需带 tool_call_id）
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response


In [ ]:
# 启动带工具的 Gradio 聊天，试问伦敦票价
gr.ChatInterface(fn=chat).launch()


## 改进方向

- **同一响应里多个 tool_calls**：一次返回多个城市查询
- **串行多轮工具**：模型看完第一次工具结果后，可能还要再调工具 → 用 `while`


In [ ]:
# ========== 多 tool_calls 版 chat：一次响应里可执行多个工具 ==========

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        # handle_tool_calls（复数）：返回多条 tool 结果
        responses = handle_tool_calls(message)
        messages.append(message)
        # extend：把多条 tool 消息依次追加
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content


In [ ]:
# ========== handle_tool_calls：遍历 message.tool_calls，逐个执行 ==========

def handle_tool_calls(message):
    responses = []
    # 可能有多个并行提议的工具调用
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                # 每个结果必须对应原来的 tool_call.id
                "tool_call_id": tool_call.id
            })
    return responses


In [ ]:
# 再开一轮 Gradio，验证多工具结果拼接
gr.ChatInterface(fn=chat).launch()


In [ ]:
# ========== while 版：允许模型连续多轮要工具 ==========

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 只要还在要工具，就继续执行→回灌→再请求
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        # 注意：循环里继续传 tools，方便下一轮再调用
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content


In [ ]:
# 导入 sqlite3：后面用本地数据库存票价，而不是写死字典
import sqlite3


In [ ]:
# ========== 建库建表：prices(city PRIMARY KEY, price REAL) ==========

# 数据库文件名（相对当前工作目录）
DB = "prices.db"

# with 连接：用完自动关闭
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    # IF NOT EXISTS：重复运行笔记本也不会报错
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()


In [ ]:
# ========== 改写 get_ticket_price：从 SQLite 读价 ==========

def get_ticket_price(city):
    # flush=True：立刻把日志打到终端，方便观察工具是否被调用
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        # 参数化查询，避免拼接 SQL；城市统一小写
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        # 有行就格式化价格；否则提示无数据
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"


In [ ]:
# 查一下 London（若尚未 seed 数据，可能返回无数据提示）
get_ticket_price("London")


In [ ]:
# ========== set_ticket_price：写入 / 更新票价 ==========

def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        # UPSERT：有则更新，无则插入
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()
        return f"Set ticket price to {city} as {price}"


In [ ]:
# ========== 种子数据：批量写入几个城市票价 ==========

ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
# 逐城调用 set_ticket_price，落到 prices.db
for city, price in ticket_prices.items():
    set_ticket_price(city, price)


In [ ]:
# 验证 Tokyo 是否写入成功
get_ticket_price("Tokyo")


In [ ]:
# 此时 chat 仍绑定旧 tools（只有 get）；先体验数据库版查询
gr.ChatInterface(fn=chat).launch()


## 练习

**任务**：再增加一个工具，让模型也能 **设置门票价格**（调用你的 `set_ticket_price`）。

思路：再写一份 function schema → 放进 `tools` → 在 `handle_tool_calls` 里按名字分支处理。


In [ ]:
# ========== 两个工具的 schema：get + set ==========

# 描述我们的函数需要一个特定的字典结构：

get_price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Get the destination city and the ticket price from user and set it to database",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer want to set the price",
            },
            "ticket_price": {
                "type": "string",
                "description": "The price ticket to destination city",
            },
        },
        "required": ["destination_city", "ticket_price"],
        "additionalProperties": False
    }
}

# 这包含在工具列表中：覆盖之前的 tools，使模型同时可见 get/set
tools = [{"type": "function", "function": get_price_function}, {"type": "function", "function": set_price_function}]


In [ ]:
# 确认 tools 里现在有两个 function
tools


In [ ]:
# ========== handle_tool_calls：按函数名分支 get / set ==========

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        # 查询票价
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        # 设置票价
        elif tool_call.function.name == "set_ticket_price":
            # 调试打印：确认走到 set 分支
            print("Here")
            arguments = json.loads(tool_call.function.arguments)
            print(arguments)
            city = arguments.get('destination_city')
            price = arguments.get('ticket_price')
            print(city, price)
            # 写入数据库
            set_price_msg = set_ticket_price(city, price)
            responses.append({
                "role": "tool",
                "content": set_price_msg,
                "tool_call_id": tool_call.id
            })
    return responses


In [ ]:
# ========== 最终 chat：while + 双工具 ==========

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content


In [ ]:
# 启动最终版助手：可查价也可改价
gr.ChatInterface(fn=chat).launch()


In [ ]:
# 不经模型，直接写库验证 set 函数本身
set_ticket_price("Vietnam", "1000")


In [ ]:
# 读回 Vietnam，确认 UPSERT 生效
get_ticket_price("Vietnam")


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">业务应用</h2>
            <span style="color:#181;">
                希望这几乎不需要解释：你已经能让 LLM <b>采取行动</b>。
                航空助手现在不止会聊天，还能通过工具与「票价数据 / 预订 API」交互——这正是客服与运营自动化的起点。
            </span>
        </td>
    </tr>
</table>
